# registry

> Defining hyperelements and collecting their styles and behaviors

In [ ]:
#| default_exp registry

A UI component needs HTML markup, CSS styling, and sometimes browser behavior. FastHTML already gives us the markup constructors. Here we build a registry to hold the styling and behaviors, associate them with their constructor, and collect everything a page needs into one stylesheet and one script. The same registry holds application styling that belongs to no component, such as page layout.

In [ ]:
import hashlib, inspect, json
from collections.abc import Mapping
from fastcore.utils import *
import fasthtml.components

In [ ]:
from fasthtml.common import Label, Input, Button, to_xml
from fastcore.test import *

## Rules

Styling and behavior both pair a *selector* that picks elements with a *payload* saying what happens to them: CSS declarations for styling, an event and handler for behavior. Because a page receives all styling as one stylesheet and all behavior as one script, we register the two together: a named rule holds CSS text and behavior records, filed under a [cascade layer](https://developer.mozilla.org/en-US/docs/Web/CSS/@layer).

The layer list is fixed and ordered. For normal declarations, each layer beats the ones before it regardless of specificity. An application can therefore restyle a built-in component from the `app` layer without changing its definition. The library never uses `!important`, partly because it reverses the layer order.


In [ ]:
LAYERS = ('reset','tokens','base','layout','components','app','utilities')

Behaviors are a dict mapping `(selector, event)` keys to JavaScript handler bodies. Each handler runs with `el` bound to the matched element and `e` to the event. For an event that doesn't bubble, such as `toggle`, add a truthy third item to its key to attach the handler in the capture phase. When a rule declares a root selector `sel`, each `&` in its behavior selectors stands for that selector, as in CSS nesting.

In [ ]:
def _behaviors(on, sel):
    "Behavior records with each `&` in a selector replaced by `:is(sel)`, to keep a selector list grouped"
    res = []
    for (selector, event, *capture), js in (on or {}).items():
        if '&' in selector:
            if sel is None: raise ValueError(f'`&` in {selector!r} needs sel')
            selector = selector.replace('&', f':is({sel})')
        res.append(dict(sel=selector, ev=event, js=js, cap=bool(capture and capture[0])))
    return res

A rule's key is a name, or `owner.name` when the rule belongs to a hyperelement. `Rule` checks the layer, expands the behaviors, and takes its `owner` and `name` from the key. A rule that declares a variant stores the variant's attribute in `variant`:


In [ ]:
class Rule:
    "A named CSS and behavior definition in a cascade layer"
    def __init__(
        self,
        key,          # Registry key: a name, or `owner.name` for a hyperelement's rule
        css='',       # Complete CSS, or a no-argument function returning it
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        if layer not in LAYERS: raise ValueError(f'Unknown layer: {layer}')
        store_attr('key,css,sel,layer,doc,variant')
        self.on = _behaviors(on, sel)

    @property
    def owner(self): return self.key.split('.', 1)[0] if '.' in self.key else None

    @property
    def name(self): return self.key.split('.', 1)[-1]

    @property
    def compiled_css(self):
        "Evaluate callable CSS and return the stylesheet text"
        return (self.css() if callable(self.css) else self.css).strip()

    def __repr__(self):
        sel = f' {self.sel}' if self.sel else ''
        doc = f': {self.doc}' if self.doc else ''
        return f'{self.key} [{self.layer}]{sel}{doc}'

`Registry.rule` takes CSS first and an optional name second. Without a name, it hashes the CSS, selector, expanded behaviors and layer into a deterministic key. Identical registrations share one entry, including behavior-only rules. Callable CSS needs an explicit name. Give a name when edits should replace the previous definition rather than add another rule. A component's rule uses `owner.name` as its key.

The registry keeps its rules in one dict. Registering a key again updates that rule in place, so existing references stay current and the rule keeps its position in the stylesheet. Each registration and removal increments `version`. `Registry.assets` rebuilds the stylesheet and script only when `version` changes. Each entry of `layers` is a live view of the rules in that layer:


In [ ]:
class RuleView(Mapping):
    "A live, read-only view of one layer or one hyperelement's rules"
    def __init__(self, registry, *, layer=None, owner=None): store_attr('registry,layer,owner')
    def _matches(self, r): return (self.layer is None or r.layer==self.layer) and (self.owner is None or r.owner==self.owner)
    def __iter__(self): return (r.name if self.owner is not None else k for k,r in self.registry.rules.items() if self._matches(r))
    def __len__(self): return sum(1 for _ in self)
    def __getitem__(self, name):
        key = f'{self.owner}.{name}' if self.owner is not None else name
        r = self.registry.rules[key]
        if not self._matches(r): raise KeyError(name)
        return r
    def __repr__(self): return '\n'.join(repr(r) for r in self.values()) or '(no rules)'

class Registry:
    "Named rules and hyperelements, from which a page's stylesheet and script are built"
    def __init__(self):
        self.rules, self.elements, self.files, self.version = {}, {}, {}, 0
        self.layers = {name: RuleView(self, layer=name) for name in LAYERS}

    def rule(
        self,
        css='',       # Complete CSS, or a no-argument function returning it
        name=None,    # Registry key; omit for a hash of the CSS, selector, behaviors and layer
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors; does not scope CSS
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        "Register CSS and behaviors, replacing an existing name in place"
        if name is None and not isinstance(css, str): raise ValueError('Callable CSS requires a name')
        r = Rule(name or '', css, on=on, sel=sel, layer=layer, doc=doc, variant=variant)
        if name is None: r.key = hashlib.sha256(json.dumps((css, sel, r.on, layer)).encode()).hexdigest()[:12]
        if r.key in self.rules: self.rules[r.key].__dict__.update(vars(r))
        else: self.rules[r.key] = r
        self.version += 1
        return self.rules[r.key]

    def remove(self, key):
        "Remove and return the rule at its full registry key"
        r = self.rules.pop(key)
        self.version += 1
        return r

    def file(
        self,
        name, # Name the file is served under
        path, # Where the file is on disk
        url,  # A public URL of the same file, which rules use to refer to it
    ):
        "Register a file that rules refer to by `url`, and return `url`. `Registry.serve` serves a copy of the file at `path` and replaces `url` with that copy's URL in the stylesheet and script"
        self.files[name] = (Path(path), url)
        return url

    def __repr__(self): return f'Registry({len(self.rules)} rules, {len(self.elements)} elements)'

    def _repr_markdown_(self):
        layers = '\n'.join(f'- `{name}`: {len(rules)} rules' for name,rules in self.layers.items())
        elements = ', '.join(f'`{k}`' for k in self.elements) or 'none'
        rules = '\n'.join(f'- `{k}` ({r.layer})' + (f': {r.doc}' if r.doc else '') for k,r in self.rules.items() if r.owner is None) or 'none'
        return f'{self!r}\n\nLayers:\n{layers}\n\nElements: {elements}\n\nRules of no element:\n{rules}'

reg = Registry()

The simplest rule is bare CSS with no constructor or explicit name, such as a centered page column. Its layer defaults to `app`. Registering the same CSS again returns the existing entry:


In [ ]:
page_css = 'main { max-width: 65ch; margin-inline: auto; padding: 1rem; }'
page_rule = reg.rule(page_css)
test_eq(page_rule.css, page_css)
test_is(reg.rule(page_css), page_rule)
page_rule

app-page [app]: A centered single-column page.

A behavior with no constructor of its own works the same way. Here's a `copy` attribute: put it on any control, and a click copies the attribute's value to the clipboard, or the element's text if the value is empty. The attribute is the natural root selector, so we declare it as `sel` and write the behavior against `&`. The rule stores the expanded behavior record:

In [ ]:
copy_rule = reg.rule(sel='[copy]', on={('&', 'click'): r"navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)"},
    doc='A control carrying `copy` copies its value, or its text, to the clipboard.')
copy_rule.on

[{'sel': ':is([copy])',
  'ev': 'click',
  'js': "navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)",
  'cap': False}]

In [ ]:
with expect_fail(ValueError): reg.rule('a { color: teal; }', layer='appp')
with expect_fail(ValueError): reg.rule(name='paste', on={('&', 'click'): ''})

The registry's own doc lists each rule that belongs to no component, such as the page column and `copy`:

In [ ]:
reg

Registry(2 rules, 0 elements)

Layers:
- `reset`: 0 rules
- `tokens`: 0 rules
- `base`: 0 rules
- `layout`: 0 rules
- `components`: 0 rules
- `app`: 2 rules
- `utilities`: 0 rules

Elements: none

Rules of no element:
- `app-page` (app): A centered single-column page.
- `copy` (app): A control carrying `copy` copies its value, or its text, to the clipboard.

## Building the stylesheet

We start with a bare `@layer` statement listing the layer names to declare their precedence. Then each layer's rules follow as one block, in registration order, with a comment naming each rule. When a rule's CSS depends on rules registered after it, give the CSS as a function. `build_css` calls that function each time it builds the stylesheet:


In [ ]:
@patch
def build_css(self:Registry):
    "Assemble the registry's CSS in cascade-layer order"
    out = [f"@layer {', '.join(LAYERS)};"]
    for layer, rules in self.layers.items():
        parts = [f'/* {r.key} */\n{css}' for r in rules.values() if (css := r.compiled_css)]
        if parts: out.append(f'@layer {layer} {{\n' + '\n'.join(parts) + '\n}')
    return '\n\n'.join(out) + '\n'

In [ ]:
print(reg.build_css())

@layer reset, tokens, base, layout, components, app, utilities;

@layer app {
/* app-page */
main { max-width: 65ch; margin-inline: auto; padding: 1rem; }
}



## Hyperelements

A component is a constructor plus rules. The constructor is a standard FastTag, or a Python function that combines several. A `Hyperelement` joins the two. Calling it calls the constructor, and its `sel` and `rules` attributes hold the root selector and a live view of the component's rules:


In [ ]:
def _nest(sel, css):
    if callable(css): return lambda: _nest(sel, css())
    return f'{sel} {{\n{css.strip()}\n}}' if css.strip() else ''
def _generic(*children, **attrs): ...

class Hyperelement:
    "A callable constructor linked to its registered rules and behaviors"
    def __init__(self, registry, name, fn, sel, doc='', generated=False):
        self.registry, self.__name__, self.fn, self.sel = registry, name, fn, sel
        self.__doc__, self.generated = doc, generated
        self.rules = RuleView(registry, owner=name)

    def __call__(self, *c, **attrs): return self.fn(*c, **attrs)

    @property
    def __signature__(self): return inspect.signature(_generic if self.generated else self.fn)

    @property
    def variants(self):
        "Declared variant attributes and their rules"
        return {r.variant: r for r in self.rules.values() if r.variant is not None}

    @property
    def behaviors(self):
        "Behavior records declared by this hyperelement's rules"
        return [b for r in self.rules.values() for b in r.on]

    def __repr__(self): return f'{self.__name__} [{self.sel}]'

    def _repr_markdown_(self):
        name = self.__name__
        sig = str(self.__signature__)
        parts = [f'### `{name}{sig}`', self.__doc__, f'Root selector: `{self.sel}`.']
        if self.variants:
            variants = [f'- `{attr}`: {r.doc}' for attr,r in self.variants.items()]
            parts.append('Declared variants:\n' + '\n'.join(variants))
        rules = [f'- `{k}` ({r.layer})' + (f': {r.doc}' if r.doc and r.doc != self.__doc__ else '') for k,r in self.rules.items() if r.variant is None]
        parts.append('Rules:\n' + '\n'.join(rules))
        return '\n\n'.join(p for p in parts if p)

An application adds rules and variants to a component through its `rule` and `variant` methods. `rule` nests the CSS it is given under the root selector, and `variant` nests it under `:is(sel)[name]`. Both register the rule with the hyperelement as its owner. A variant needs no declaration beyond its CSS, because `variant` also records the attribute:

In [ ]:
@patch
def rule(
    self:Hyperelement,
    name,         # Local rule name; registering it again replaces that rule
    css='',       # CSS declarations and nested rules, scoped to the root selector
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the root selector
    layer='app',  # One of LAYERS
    doc='',       # Description of the customization
):
    "Register CSS and behaviors scoped to this hyperelement"
    return self.registry.rule(_nest(self.sel, css), f'{self.__name__}.{name}', on=on, sel=self.sel, layer=layer, doc=doc)

@patch
def variant(
    self:Hyperelement,
    name,         # Attribute selecting the variant; also its local rule name
    css='',       # CSS declarations and nested rules, scoped to the variant
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the variant selector
    layer='app',  # One of LAYERS
    doc='',       # Description shown with the variant
):
    "Declare an attribute variant without changing the constructor"
    if name=='base': raise ValueError('base is reserved for the constructor definition')
    sel = f':is({self.sel})[{name}]'
    return self.registry.rule(_nest(sel, css), f'{self.__name__}.{name}', on=on, sel=sel, layer=layer, doc=doc, variant=name)

`hypelt` registers a component from a FastHTML tag name or a handwritten constructor. It nests `css` under the component's root selector as the `base` rule, just like `Hyperelement.rule`. Write declarations for the root, nested selectors for its children, and `&` for states or pseudo-elements on the root. A no-argument function may supply the CSS when it depends on later registrations. Global rules such as `@keyframes` belong in `reg.rule`, which accepts complete CSS without wrapping. Its `sel` parameter applies to behaviors only. The module-level `hypelt` registers with `reg`:


In [ ]:
@patch
def hypelt(
    self:Registry,
    name=None,   # FastHTML tag name or a constructor; omit to use as a decorator
    *,
    sel=None,    # Root selector; required for a handwritten constructor
    css='',      # CSS declarations and nested rules, or a no-argument function returning them
    on=None,     # Mapping of (selector, event[, capture]) to JavaScript handler bodies
    layer='app', # One of LAYERS; built-ins use components, layout, or base
    doc=None,    # Description; defaults to a handwritten constructor's docstring
):
    "Register a constructor and its base rule, returning a callable hyperelement"
    if name is None: return partial(self.hypelt, sel=sel, css=css, on=on, layer=layer, doc=doc)
    generated = isinstance(name, str)
    if generated:
        fn = getattr(fasthtml.components, name)
        if sel is None: sel = fn().tag
    else:
        fn, name = name, name.__name__
        if sel is None: raise ValueError('A handwritten constructor needs sel')
        if doc is None: doc = fn.__doc__
    if name in self.elements and self.elements[name].sel != sel:
        raise ValueError(f'{name} is registered with root selector {self.elements[name].sel!r}, which its rules and variants target; register a new hyperelement for a new root')
    self.rule(_nest(sel, css), f'{name}.base', on=on, sel=sel, layer=layer, doc=doc or '')
    obj = Hyperelement(self, name, fn, sel, doc or '', generated)
    if name in self.elements: self.elements[name].__dict__.update(vars(obj))
    else: self.elements[name] = obj
    return self.elements[name]

hypelt = reg.hypelt

As an example component, let's make a dismissible notice — a box announcing something like "changes saved", with a button to close it. HTML has no element for that, so we use a custom tag, `notice-box`. Clicking any child marked `dismiss` removes the notice:

In [ ]:
notice_css = r"""
display: block;
padding: .75rem 1rem;
background: #f4f4f6;
border: 1px solid #ccc;
border-radius: 8px;
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, on={('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"},
    layer='components', doc='A dismissible notice.')
to_xml(NoticeBox('Changes saved.', Button('Dismiss', dismiss=True)))

'<notice-box>Changes saved.<button dismiss>Dismiss</button></notice-box>'

FastHTML generated the constructor from the CamelCase name. `hypelt` called it with no arguments and used the empty element's tag as the root selector. The hyperelement's view and the registry hold the same rule object:


In [ ]:
test_eq(NoticeBox().tag, NoticeBox.sel)
assert NoticeBox.rules['base'] is reg.rules['NoticeBox.base']
assert NoticeBox.rules['base'].compiled_css.startswith('notice-box {')
NoticeBox

### `NoticeBox(*children, **attrs)`

A dismissible notice.

Root selector: `notice-box`.

Rules:
- `base` (components)

The second case is a handwritten constructor. Consider a labelled field. It uses native `label` and `input` elements, with the input wrapped in its label so the two are associated without an id. Our function, `Field`, needs arguments, so `hypelt` can't call it with none to read a selector. Instead we mark the label with a `field` attribute and supply `label[field]` as `sel`, using the decorator form:

In [ ]:
field_css = r"""
display: grid;
gap: .5rem;
"""

@hypelt(sel='label[field]', css=field_css)
def Field(label, **attrs):
    "A labelled input"
    return Label(label, Input(**attrs), field=True)

to_xml(Field('Email', name='email', type='email'))

'<label field>Email  <input name="email" type="email">\n</label>'

## Customization

There are two ways to change how a component looks or behaves. An override is separate CSS that you register in the `app` layer, without changing the component's definition. A redefinition is a second `hypelt` call with the same name, whose constructor and base rule replace the old ones completely.

### Overrides

Say a developer wants rounder notices in their application. `NoticeBox.rule` registers a named rule in the `app` layer by default, where it beats the component's own CSS in the `components` layer:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notices in this application.')


NoticeBox.corners [app] notice-box: Rounder notices in this application.

To select a style with an attribute, declare the attribute as a variant. The variant's description then appears in the hyperelement's documentation, and the same constructor accepts the attribute:


In [ ]:
NoticeBox.variant('quiet', 'background: transparent; border-color: transparent;', doc='An unfilled notice without a visible border.')


NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.

In [ ]:
test_eq(list(NoticeBox.variants), ['quiet'])
to_xml(NoticeBox('Saved quietly.', quiet=True))

'<notice-box quiet>Saved quietly.</notice-box>'

To change a component only within part of a page, register an application rule with a selector for that part. Here `editor-notices` squares the corners of every notice inside an `[editor]` element. The optional second argument gives it a name for later replacement. It doesn't change the notice's definition, and it doesn't appear in `NoticeBox.rules`, which lists only the rules registered on the component. The `app` layer's view lists it with the other application rules:


In [ ]:
reg.rule('[editor] notice-box { border-radius: 0; }', 'editor-notices', doc='Square notices inside the editor.')
reg.layers['app']

app-page [app]: A centered single-column page.
copy [app] [copy]: A control carrying `copy` copies its value, or its text, to the clipboard.
Field.base [app] label[field]: A labelled input
NoticeBox.corners [app] notice-box: Rounder notices in this application.
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.

### Redefinitions

Registering a hyperelement again updates its constructor and base rule in place. References held elsewhere now point to the new definition. The component's application rules and declared variants stay registered. Omitted CSS or behaviors are cleared rather than merged, since the new call is the whole definition. `hypelt` rejects a redefinition with a different root selector, because the component's rules and variants are nested under the original one. Here the redefinition has no `on`, so the notice becomes persistent:


In [ ]:
original, base = NoticeBox, NoticeBox.rules['base']
NoticeBox = hypelt('NoticeBox', css=lambda: notice_css.replace('8px', '12px'), layer='components', doc='A persistent notice.')
test_is(NoticeBox, original)
test_is(NoticeBox.rules['base'], base)
assert '12px' in base.compiled_css
assert 'corners' in NoticeBox.rules and 'quiet' in NoticeBox.variants
with expect_fail(ValueError): hypelt('NoticeBox', sel='notice-box[persistent]', css=notice_css, layer='components')
NoticeBox

### `NoticeBox(*children, **attrs)`

A persistent notice.

Root selector: `notice-box`.

Declared variants:
- `quiet`: An unfilled notice without a visible border.

Rules:
- `base` (components)
- `corners` (app): Rounder notices in this application.

The base rule now declares no behavior. To remove the application override, pass its full key to `reg.remove`. A layer view taken earlier no longer lists it:


In [ ]:
test_eq(NoticeBox.behaviors, [])
app_rules = reg.layers['app']
v = reg.version
reg.remove('NoticeBox.corners')

NoticeBox.corners [app] notice-box: Rounder notices in this application.

In [ ]:
assert 'NoticeBox.corners' not in app_rules
test_eq(reg.version, v+1)
app_rules

app-page [app]: A centered single-column page.
copy [app] [copy]: A control carrying `copy` copies its value, or its text, to the clipboard.
Field.base [app] label[field]: A labelled input
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()